# 08 — Experiment 4c (DPO)

**Variable under test:** preference learning on top of Exp 4b SFT checkpoint.

**Config differences from Exp 4b:**
- SFTTrainer → DPOTrainer
- Loss: cross-entropy imitation → DPO preference loss
- LR: 2e-4 → 5e-6 (~40x lower, standard for DPO)
- Epochs: 5 → 2 (DPO converges faster)
- New: preference pair dataset (75-100 pairs, hard negatives + synthetic perturbations)

**What to watch for:**
- No regression on Exp 4b's 100% scorecard (marker / slot / banned-adjective)
- Positional-schema correctness on stacked bars (the Apr 20 falsified prediction)
- Internal consistency check (no "tooltip visible" / "no tooltip" contradictions)
- DPO loss descends below the random-preference baseline (ln(2) ≈ 0.69)

**Starting from:** `/mnt/civicinsight/checkpoints/exp4-visionunfrozen/checkpoint-65/` (Exp 4b SFT adapter)  
**Saving to:** `/mnt/civicinsight/checkpoints/exp4c-dpo/`

In [ ]:
# ensure new Volume paths exist
!mkdir -p /mnt/civicinsight/checkpoints/exp4c-dpo /mnt/civicinsight/results
!ls -la /mnt/civicinsight/

In [ ]:
%%capture
%uv pip install unsloth
%uv pip install pillow==11.3.0
%uv pip install --upgrade transformers trl

In [ ]:
# imports
from unsloth import FastVisionModel          # MUST be first
from trl import DPOTrainer, DPOConfig
import torch
import os
import json
import re
import random
from PIL import Image
from huggingface_hub import login
import time

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
random.seed(42)  # reproducible synthetic perturbations

In [ ]:
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via Modal Secret")
else:
    print("HF_TOKEN not found in environment. Add it as a Modal Secret.")

In [ ]:
# load base Gemma 4 + apply Exp 4b SFT adapter as DPO starting point
SFT_CHECKPOINT = "/mnt/civicinsight/checkpoints/exp4-visionunfrozen/checkpoint-65"

model, tokenizer = FastVisionModel.from_pretrained(
    "/mnt/civicinsight/model",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Base model loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# apply the Exp 4b LoRA adapter on top
model.load_adapter(SFT_CHECKPOINT)
print(f"\nApplied SFT adapter from: {SFT_CHECKPOINT}")
print(model.print_trainable_parameters())  # expect ~39.4M trainable (matches Exp 4b)

In [ ]:
# perturbation library — inline functions per CLAUDE.md (no /src/ imports)
# each takes a gold aria_label and returns a hard-negative perturbation

MARKER = "[civicinsight-v1]"
BANNED_HEDGES = ["around ", "approximately ", "roughly ", "about "]

def strip_marker(gold):
    """Drop the [civicinsight-v1] marker. Targets marker rule."""
    return gold.replace(f"{MARKER} ", "").replace(MARKER, "").strip()

def wrong_slot(gold):
    """Replace 'This [chart_type]' with drift patterns from Exp 4 documentation."""
    if "This " not in gold:
        return gold  # no-op if pattern not present
    drift = random.choice([
        lambda s: s.replace("This ", "A ", 1),                        # English default prior
        lambda s: re.sub(r"This (\w+)", r"\1", s, count=1),            # no article
        lambda s: s.replace("This ", "Bar chart showing ", 1),        # different verb
    ])
    return drift(gold)

def inject_hedge(gold):
    """Prefix the first standalone number with a banned hedge."""
    match = re.search(r"\b(\d{1,3}(?:[ ,]\d{3})*(?:\.\d+)?)\b", gold)
    if not match:
        return gold
    hedge = random.choice(BANNED_HEDGES)
    num = match.group(1)
    return gold.replace(num, f"{hedge}{num}", 1)

def positional_schema_fill(gold):
    """Swap the first two percentage values in the output — simulates positional misbinding.
    Targets the Apr 20 falsified prediction (vision unfreeze didn't fix this)."""
    matches = list(re.finditer(r"\b(\d{1,3})%", gold))
    if len(matches) < 2:
        return gold
    m1, m2 = matches[0], matches[1]
    v1, v2 = m1.group(1), m2.group(1)
    return gold[:m1.start()] + v2 + gold[m1.end():m2.start()] + v1 + gold[m2.end():]

def break_consistency(gold):
    """Append a contradictory tail sentence. Targets internal consistency."""
    if "tooltip" in gold.lower() or "selected" in gold.lower():
        return gold.rstrip(".") + ". No tooltip is visible and no item is selected."
    return gold.rstrip(".") + ". The chart is empty."

def googly_wrap(gold):
    """Wrap in **bold** + bullets. Targets style regression (easy negative — use sparingly)."""
    body = gold.replace(MARKER, "").strip()
    return f"**Aria Label:**\n\n* {body}\n\nThis description is provided for screen readers."

PERTURBATIONS = {
    "strip_marker": strip_marker,
    "wrong_slot": wrong_slot,
    "inject_hedge": inject_hedge,
    "positional_schema_fill": positional_schema_fill,
    "break_consistency": break_consistency,
    "googly_wrap": googly_wrap,
}

# sanity check: apply each perturbation to a representative gold
_sample = "[civicinsight-v1] This stacked bar chart shows USA at 80% urban and 19% rural areas."
print(f"GOLD: {_sample}\n")
for name, fn in PERTURBATIONS.items():
    print(f"[{name}]")
    print(f"  {fn(_sample)}\n")

In [ ]:
# build preference pair dataset (target: 75-100 pairs)

DATASET_JSON = "/mnt/civicinsight/training/dataset.marked.json"
STD_TRAIN_IMAGE_DIR = "/mnt/civicinsight/standardized"
STD_TEST_IMG_DIR = "/mnt/civicinsight/test"
HELDOUT_GOLDS_PATH = "/mnt/civicinsight/training/heldout-golds.json"  # optional

PROMPT = "Generate an aria-label for this data visualization image."

# load training golds (50 examples)
with open(DATASET_JSON) as f:
    dataset = json.load(f)

# load held-out golds if available (user-written, optional)
heldout_golds = {}
if os.path.exists(HELDOUT_GOLDS_PATH):
    with open(HELDOUT_GOLDS_PATH) as f:
        heldout_golds = json.load(f)
    print(f"Loaded {len(heldout_golds)} held-out gold aria_labels")
else:
    print(f"No heldout-golds.json found at {HELDOUT_GOLDS_PATH}")
    print("Proceeding with synthetic-only preference pairs (no real-failure pairs)")

def make_pair(image, gold, rejected, prompt=PROMPT):
    """Build a single DPO preference pair record (vision-multimodal format)."""
    return {
        "prompt": [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]}
        ],
        "chosen": [{"role": "assistant", "content": [{"type": "text", "text": gold}]}],
        "rejected": [{"role": "assistant", "content": [{"type": "text", "text": rejected}]}],
    }

preference_pairs = []

# ---- synthetic pairs from training golds ----
# rotate through perturbation library; over-represent positional_schema_fill on stacked bars
ROTATION = ["strip_marker", "wrong_slot", "inject_hedge", "break_consistency", "positional_schema_fill"]

synthetic_count = 0
for i, record in enumerate(dataset):
    img_path = os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"]))
    img = Image.open(img_path)
    gold = record["aria_label"]
    
    # primary perturbation: rotate through library for coverage
    pert_name = ROTATION[i % len(ROTATION)]
    rejected = PERTURBATIONS[pert_name](gold)
    if rejected != gold:  # skip if perturbation was a no-op
        preference_pairs.append(make_pair(img, gold, rejected))
        synthetic_count += 1
    
    # extra positional-schema perturbation for stacked bars (over-represent)
    if ("stacked" in gold.lower() or "bar chart" in gold.lower()) and "%" in gold:
        rejected_ps = PERTURBATIONS["positional_schema_fill"](gold)
        if rejected_ps != gold:
            preference_pairs.append(make_pair(img, gold, rejected_ps))
            synthetic_count += 1

# ---- real-failure pairs from held-out images (only if user wrote heldout-golds.json) ----
# captured failing outputs parsed from docs/exp4-results.md, exp4-reproducibility-0421.md, exp4b-results.md
HELDOUT_FAILURES = {
    "baseline-1.png": [
        # Exp 4 (frozen): French drift, no marker
        "Carte chorop\u00e8the repr\u00e9sentant les prix immobiliers en France selon les arrondissements.",
        # Exp 4b (vision unfrozen): geographic overreach to 'France', misspelled commune
        "[civicinsight-v1] This choropleth map shows real estate prices per square metre in arrondissements of France. The color scale ranges from 0 - 1076 euros per square metre (light pink) to 2989 - 6571 euros per square metre (dark red). The tooltip for the Pl\u0153renq Arrondissement shows a price of 2643 EUR/m2.",
    ],
    "browser-share-other-filtered.png": [
        # Exp 4: Opera hallucinated, filter state missed
        "A line chart titled 'The rise of Google Chrome' showing browser market share. Around 5% of users use Opera in 2020.",
    ],
    "browser-share.png": [
        # Exp 4: Chrome/IE swap, 'around X%' leaked
        "A line chart titled 'The rise of Google Chrome'. Around 5% of users in 2009 used Chrome, generally trends upward steadily.",
    ],
    "income-vs-life-exp.png": [
        # Exp 4: scatter mode-collapse loop
        "A scatter plot showing GDP per capita and life expectancy. Samoa near 10k per capita and 75 years life expectancy. Tonga near 10k per capita and 75 years life expectancy. Samoa near 10k per capita.",
    ],
    "rural-vs-urban.png": [
        # Exp 4: misclassified as Bar chart (not stacked), USA '1' hallucinated, Reykjavik diacritic dropped
        "Bar chart showing rural and urban population. USA has 1% capital share. Reykjavik has 56%. Argentina at 35%.",
    ],
}

real_pair_count = 0
if heldout_golds:
    for img_name, gold in heldout_golds.items():
        img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
        if not os.path.exists(img_path):
            print(f"Skipping {img_name}: image file not found at {img_path}")
            continue
        img = Image.open(img_path)
        for failure in HELDOUT_FAILURES.get(img_name, []):
            preference_pairs.append(make_pair(img, gold, failure))
            real_pair_count += 1

print(f"\nTotal preference pairs: {len(preference_pairs)}")
print(f"  Synthetic (perturbations of training golds): {synthetic_count}")
print(f"  Real (held-out failures vs hand-written golds): {real_pair_count}")

# sanity print of one pair (text only, image truncated)
if preference_pairs:
    sample = preference_pairs[0]
    print("\n=== Sample pair ===")
    print(f"Prompt text: {sample['prompt'][0]['content'][1]['text']}")
    print(f"Chosen:   {sample['chosen'][0]['content'][0]['text'][:200]}...")
    print(f"Rejected: {sample['rejected'][0]['content'][0]['text'][:200]}...")

In [ ]:
# ============================================================
# BEFORE-DPO baseline — capture model state with SFT adapter only
# Comparison anchor: DPO either improves on these or it doesn't
# do_sample=False for deterministic comparison (per docs/exp4-reproducibility-0421.md)
# ============================================================
model.eval()

test_images = sorted([f for f in os.listdir(STD_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images for before-DPO sweep\n")

before_dpo_results = {}
for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT},
    ]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"BEFORE-DPO | {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

model.train()  # back to train mode before DPO

In [ ]:
# ============================================================
# DPO training run — the main event
# Hyperparameters:
#   beta=0.1   — KL penalty (standard DPO starting point)
#   LR=5e-6    — ~40x lower than SFT's 2e-4 (DPO requires lower LR)
#   epochs=2   — DPO converges faster than SFT (1-3 typical)
# NOTE: vision DPO is less battle-tested than text-only. If DPOTrainer init fails,
# check TRL version + Unsloth's vision DPO docs (they may have a Unsloth-specific wrapper).
# ============================================================

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                          # auto-creates frozen reference from current model state
    tokenizer=tokenizer,                     # may need 'processing_class=tokenizer' on newer TRL (≥0.20)
    train_dataset=preference_pairs,
    args=DPOConfig(
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=5e-6,
        output_dir="/mnt/civicinsight/checkpoints/exp4c-dpo",
        max_length=2048,
        max_prompt_length=1024,
        remove_unused_columns=False,         # required for image+text
        logging_steps=1,                     # log every step (small dataset)
    ),
)

print(f"GPU before DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"Total preference pairs: {len(preference_pairs)}")
print(f"\nStarting DPO training...")
dpo_trainer.train()
print("\nDPO training ran without crash!")
print(f"GPU after DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ============================================================
# AFTER-DPO held-out sweep — same 5 images, same prompt, same do_sample=False
# Direct comparison anchor against before_dpo_results
# ============================================================
model.eval()

after_dpo_results = {}
for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT},
    ]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    after_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"AFTER-DPO | {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

In [ ]:
# ============================================================
# Extended scorecard — existing 3 metrics + 2 NEW (positional-schema, consistency)
# Compares before-DPO vs after-DPO across all 5 held-outs
# ============================================================

MARKER = "[civicinsight-v1]"
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

# NEW: positional-schema known-wrong segment-value bindings per held-out image
# format: list of (label_substr, wrong_value_substr) — if BOTH appear within 100 chars, flag
POSITIONAL_KNOWN_WRONGS = {
    "rural-vs-urban.png": [
        ("USA", "80%"),       # USA capital ~10%; the 80% is on other-urban
        ("Australia", "88%"), # Australia capital ~8%; the 88% is on other-urban
        ("China", "53%"),     # China capital ~7-10%; the 53% is on other-urban
    ],
}

# NEW: internal-consistency contradiction patterns
CONSISTENCY_CONTRADICTIONS = [
    (r"tooltip\s+is\s+visible", r"no\s+tooltip\s+is\s+visible"),
    (r"is\s+selected", r"no\s+\w+\s+is\s+selected"),
]

def score(output, img_name=None):
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    lower = assistant_part.lower()
    
    has_marker = MARKER in assistant_part
    opens_with_slot = any(
        assistant_part.lstrip().startswith(MARKER + " " + s)
        or assistant_part.lstrip().startswith(s)
        for s in SLOT_OPENERS
    )
    banned_found = [a.strip() for a in BANNED_ADJECTIVES if a in lower]
    
    # NEW: positional-schema correctness (only meaningful for stacked bar held-outs)
    positional_errors = []
    if img_name and img_name in POSITIONAL_KNOWN_WRONGS:
        for s1, s2 in POSITIONAL_KNOWN_WRONGS[img_name]:
            if s1 in assistant_part and s2 in assistant_part:
                idx1 = assistant_part.index(s1)
                idx2 = assistant_part.index(s2)
                if abs(idx1 - idx2) < 100:
                    positional_errors.append(f"{s1}+{s2}")
    
    # NEW: internal consistency — contradiction detection
    consistency_violations = []
    for p1, p2 in CONSISTENCY_CONTRADICTIONS:
        if re.search(p1, lower) and re.search(p2, lower):
            consistency_violations.append(f"{p1} AND {p2}")
    
    return {
        "has_marker": has_marker,
        "opens_with_slot": opens_with_slot,
        "banned_adjectives_found": banned_found,
        "positional_errors": positional_errors,
        "consistency_violations": consistency_violations,
    }

def print_scorecard(results, label):
    print(f"\n{'='*100}")
    print(f"SCORECARD — {label}")
    print(f"{'='*100}")
    print(f"{'image':<45} {'marker':<8} {'slot':<6} {'banned':<25} {'pos-err':<15} {'consist'}")
    print("-" * 100)
    for img_name, out in results.items():
        s = score(out, img_name)
        banned = ",".join(s["banned_adjectives_found"]) or "none"
        pos = ",".join(s["positional_errors"]) or "none"
        consist = "FAIL" if s["consistency_violations"] else "ok"
        print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<6} {banned:<25} {pos:<15} {consist}")
    
    n = len(results)
    marker_rate = sum(1 for o in results.values() if score(o)['has_marker']) / n
    slot_rate = sum(1 for o in results.values() if score(o)['opens_with_slot']) / n
    no_adj_rate = sum(1 for o in results.values() if not score(o)['banned_adjectives_found']) / n
    no_pos_err = sum(1 for img_name, o in results.items() if not score(o, img_name)['positional_errors']) / n
    no_consist_viol = sum(1 for o in results.values() if not score(o)['consistency_violations']) / n
    print(f"\nAggregate ({n} held-outs):")
    print(f"  marker:                 {marker_rate:.0%}")
    print(f"  slot opener:            {slot_rate:.0%}")
    print(f"  no banned adjectives:   {no_adj_rate:.0%}")
    print(f"  no positional errors:   {no_pos_err:.0%}    [NEW]")
    print(f"  no consistency issues:  {no_consist_viol:.0%}    [NEW]")

print_scorecard(before_dpo_results, "BEFORE DPO (Exp 4b SFT only)")
print_scorecard(after_dpo_results, "AFTER DPO (Exp 4c SFT+DPO)")

In [ ]:
# Save full results JSON to Modal Volume (mirror Exp 4b cell 16 pattern)
artifact = {
    "experiment": "exp4c-dpo",
    "date": "2026-04-23",
    "config": {
        "starting_checkpoint": SFT_CHECKPOINT,
        "beta": 0.1,
        "lr": 5e-6,
        "epochs": 2,
        "preference_pair_count": len(preference_pairs),
        "synthetic_count": synthetic_count,
        "real_pair_count": real_pair_count,
    },
    "before_dpo": before_dpo_results,
    "after_dpo": after_dpo_results,
}

os.makedirs("/mnt/civicinsight/results", exist_ok=True)
with open("/mnt/civicinsight/results/exp4c-results.json", "w") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)
print("Saved to /mnt/civicinsight/results/exp4c-results.json")

In [ ]:
# Push DPO adapter to HF private repo as a new commit
from huggingface_hub import HfApi

REPO_ID = "shahfazal/civicinsight-gemma4-e4b-it"
DPO_CHECKPOINT_DIR = "/mnt/civicinsight/checkpoints/exp4c-dpo"

# find the most recent checkpoint dir
checkpoint_dirs = sorted(
    [d for d in os.listdir(DPO_CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1]),
)
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoint dirs found in {DPO_CHECKPOINT_DIR}")

final_checkpoint = os.path.join(DPO_CHECKPOINT_DIR, checkpoint_dirs[-1])
print(f"Pushing checkpoint: {final_checkpoint}")

# fix base_model metadata in README before push (same workaround as Exp 4b)
readme_path = os.path.join(final_checkpoint, "README.md")
if os.path.exists(readme_path):
    with open(readme_path) as f:
        content = f.read()
    content = content.replace(
        "base_model: /mnt/civicinsight/model",
        "base_model: unsloth/gemma-4-E4B-it",
    )
    with open(readme_path, "w") as f:
        f.write(content)

api = HfApi()
api.upload_folder(
    folder_path=final_checkpoint,
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["optimizer.pt", "scheduler.pt", "rng_state.pth"],
    commit_message="Exp 4c DPO — preference learning on Exp 4b base",
)
print(f"\nPushed to https://huggingface.co/{REPO_ID}")